# Stage3 / 01 — Verify inherited assets + checkpoints + remaps

Before committing GPU time to stage3 training, confirm that:
- the chosen stage3 YAML points at a checkpoint that exists,
- its class protocol + NC are internally consistent,
- loading the stage3 model + warm-start chain does not fail.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, sys, glob
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT); sys.path.insert(0, REPO_ROOT)
!pip install -q yacs tqdm

In [ ]:
from lib.config import cfg
from stage2.lib.models.yolopv2_detrlane import get_net_yolopv2_detrlane
from stage2.lib.utils.warm_start import warm_start_from_stage1, print_warm_start_report
import torch

# Pick the integrated YAML written by notebook 00.
candidates = sorted(glob.glob(os.path.join(REPO_ROOT, 'stage3', 'configs', 'integrated_from_*.yaml')))
assert candidates, 'Run stage3/00 first.'
YAML = candidates[-1]
print('Using', YAML)

cfg.defrost(); cfg.merge_from_file(YAML); cfg.freeze()
print(f'NC={cfg.MODEL.NC}  CLASS_PROTOCOL={cfg.DATASET.CLASS_PROTOCOL}')
print(f'WARM_START_CKPT = {cfg.STAGE2.WARM_START_CKPT}')
assert os.path.exists(cfg.STAGE2.WARM_START_CKPT), 'warm-start ckpt missing'

model = get_net_yolopv2_detrlane(cfg).cpu()
report = warm_start_from_stage1(model, cfg.STAGE2.WARM_START_CKPT,
                                student_nc=int(cfg.MODEL.NC),
                                teacher_nc=int(cfg.STAGE2.WARM_START_TEACHER_NC))
print_warm_start_report(report)